<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week2_Day3_Structured_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Day 3: Structured extraction with Gemini

Deliverable: one abstract yields schema-valid claims, and missing items are not guessed.

`src/pubmed_fetch/extract.py` sends one abstract to Gemini with structured output (`ExtractedClaims`), then attaches the PMID itself. Every claim then gets automatic evidence checks (`graph_basics.evidence_checks`) and a manual review, and the whole run is saved as one JSON record (`runlog.py`).

**The automatic checks are literal.** They show whether the quote is in the abstract and whether the cell type, disease and marker names are in the quote. They do not show that the quote *supports* the claim; that is what the manual review is for.

Before running: add your Gemini key in Colab **Secrets** as `GOOGLE_API_KEY` and allow this notebook to access it.

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "Guliko24"
REPO_NAME = "PubMed_Fetch"
BRANCH = "main"  # set to a branch name to test it before merging

repo_dir = Path("/content") / REPO_NAME
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args], check=True, capture_output=True, text=True
    ).stdout.strip()

if not repo_dir.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(repo_dir)], check=True)
else:
    # /content survives "Restart session", so an older clone may be on another branch.
    git("fetch", "origin", BRANCH)
    git("checkout", BRANCH)
    git("pull", "--ff-only", "origin", BRANCH)

print(f"Clone at {repo_dir} is on branch {git('branch', '--show-current')}, commit {git('rev-parse', '--short', 'HEAD')}")

%pip -q install -r {repo_dir}/requirements.txt

# Until pyproject.toml arrives in Week 5, put src/ on the import path by hand.
for path in (repo_dir, repo_dir / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import search_utils
from pubmed_fetch import extract, graph_basics, runlog, schemas
print("Package imported from:", extract.__file__)

In [ ]:
import os

# Colab Secrets are not environment variables: copy the key into one,
# so the same code also works with a .env file on a laptop.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # not in Colab: extract.get_llm() reads .env instead

# Never print the key itself: notebook outputs are pushed to a public repo.
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))

## 1. Choose a model

Model names change often, so this asks your key which Flash models it can use. The next cell uses `gemini-3.6-flash`, the model of the first Day 3 run. If it is retired, copy a listed name (without the `models/` prefix) instead; prefer one not marked `preview`.

In [ ]:
from google import genai

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
for model in client.models.list():
    if "flash" in model.name and "generateContent" in (model.supported_actions or []):
        print(model.name)

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"  # used for the first Day 3 run; replace with a listed name if retired

assert GEMINI_MODEL, "Set GEMINI_MODEL to one of the names listed above."
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
llm = extract.get_llm()
print("Using model:", GEMINI_MODEL)

## 2. Test cases

| Label | Input | What it tests |
|---|---|---|
| `real_msn` | PMID 34582785 | Receptor names as markers (D1, D2 with gene symbols); claims without a disease |
| `real_hypoxia` | PMID 40585969 | Grouped cell types; disease named outside the quotes |
| `grouped` | synthetic | A group ("pericytes and fibroblasts") is split without strengthening |
| `negation` | synthetic | "did not" survives into the claim |
| `html` | synthetic | A quote that crosses an `<i>` tag; no disease imported from outside knowledge |
| `empty` | synthetic | No cell type named, so no claims |

In [ ]:
documents = search_utils.load_pubmed_data(repo_dir / "pubmed_abstracts.json")
abstracts = {doc["pmid"]: doc["abstract"] for doc in documents}

CASES = [
    {
        "label": "real_msn",
        "pmid": "34582785",
        "abstract": abstracts["34582785"],
        "expect": "D1 and D2 as markers with gene symbols DRD1 and DRD2; disease only where the quote names one",
    },
    {
        "label": "real_hypoxia",
        "pmid": "40585969",
        "abstract": abstracts["40585969"],
        "expect": "pericytes and fibroblasts as separate claims, no stronger than the group statement; disease null unless in the quote",
    },
    {
        "label": "grouped",
        "pmid": "synthetic-grouped",
        "abstract": "Non-neural cells, including pericytes and fibroblasts, showed a persistent response after the intervention.",
        "expect": "2 claims (pericytes, fibroblasts), each saying only that the group showed a persistent response",
    },
    {
        "label": "negation",
        "pmid": "synthetic-negation",
        "abstract": "Treatment did not significantly alter SST expression in cortical interneurons.",
        "expect": "1 claim that keeps 'did not'; no positive effect claimed",
    },
    {
        "label": "html",
        "pmid": "synthetic-html",
        "abstract": "<b>Results:</b> Parvalbumin-expressing interneurons were reduced in the <i>Scn1a</i> mutant cortex.",
        "expect": "quote keeps the <i> tags and is found; Parvalbumin -> PVALB; disease null (none is named)",
    },
    {
        "label": "empty",
        "pmid": "synthetic-empty",
        "abstract": "This review summarises statistical methods for analysing sequencing data and discusses open problems in study design.",
        "expect": "0 claims",
    },
]
print(f"{len(CASES)} cases ready.")

## 3. Extract and check

One API call per case (6 calls).

In [ ]:
results = []
for case in CASES:
    extraction = extract.extract_claims(case["pmid"], case["abstract"], llm=llm)
    checks = [graph_basics.evidence_checks(claim, case["abstract"]) for claim in extraction.claims]
    results.append({**case, "extraction": extraction, "checks": checks})
    print(f"{case['label']}: {len(extraction.claims)} claims")

In [ ]:
def show(result):
    claims = result["extraction"].claims
    print(f"=== {result['label']} ({result['pmid']}): {len(claims)} claims")
    print(f"    expected: {result['expect']}\n")
    for number, (claim, check) in enumerate(zip(claims, result["checks"]), start=1):
        markers = ", ".join(f"{m.as_written} -> {m.gene_symbol} ({m.symbol_status})" for m in claim.marker_genes) or "-"
        print(f"[{result['label']}#{number}] automatic checks passed: {check['automatic_checks_passed']}")
        print(
            f"    quote found: {check['quote_found']} | cell type in quote: {check['cell_type_in_quote']}"
            f" | disease in quote: {check['disease_in_quote']} | markers in quote: {check['markers_in_quote']}"
        )
        print(f"    cell_type:    {claim.cell_type}")
        print(f"    marker_genes: {markers}")
        print(f"    disease:      {claim.disease}")
        print(f"    claim_text:   {claim.claim_text}")
        print(f"    quote:        {claim.supporting_quote}\n")


for result in results:
    show(result)

## 4. Manual review

Read each claim above against its quote, then record a status here: `accepted`, `needs_review` or `rejected`, with a short note. The key is the label and number printed above, e.g. `real_hypoxia#4`. Claims you leave out are saved as `unreviewed`.

Ask of each claim: does the quote alone support everything the claim says? A claim can pass every automatic check and still fail this.

In [ ]:
REVIEW = {
    # "real_msn#1": ("accepted", ""),
    # "real_hypoxia#4": ("needs_review", "split from a grouped statement; check wording"),
}

automatic_failures = [
    f"{r['label']}#{n}"
    for r in results
    for n, check in enumerate(r["checks"], start=1)
    if not check["automatic_checks_passed"]
]
total = sum(len(r["checks"]) for r in results)
print(f"{total} claims; {len(REVIEW)} reviewed by hand.")
print("Failed an automatic check (review these first):", automatic_failures or "none")

## 5. Save the run record

Saves model, UTC time, git commit, package versions, prompt, inputs, claims, checks and your review as one new JSON file, then downloads it. Files in `/content` disappear with the runtime, so keep the download; it can be added to the repo under `results/`.

In [ ]:
record = runlog.make_run_record(
    model=GEMINI_MODEL,
    prompt=extract.EXTRACTION_INSTRUCTIONS,
    results=results,
    review=REVIEW,
    repo_dir=repo_dir,
)
path = runlog.save_run_record(record, repo_dir / "results" / "week2_day3")
print("Saved:", path.name, "| commit:", record["git_commit"], "| packages:", record["packages"])

try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    pass  # not in Colab: the file is already on disk

## What to check

- Every call returns without a validation error: the output is schema-valid.
- Compare each case with its `expected` line. Note every mismatch; they are Day 5 material.
- `empty` must return 0 claims.
- Any claim that fails an automatic check gets a manual status before saving.
- One run is an observation, not a measured accuracy: `gemini-3.6-flash` ignores `temperature`, so runs vary.